# Landing → Bronze

Ingestão das 8 tabelas num notebook só (mesma lógica pra todas: ler fiel à origem, escrever com
`overwrite`) — muda só por PR revisado, igual ao notebook base em `analises/`. No workspace do grupo
este notebook fica em `Shared/case-final-rocket-lab/medalhao/Landing_To_Bronze` (ver
`docs/databricks-estrutura.md`).

**Os CSVs ficam num Volume do Unity Catalog** (`dados_vcredit`, caminho completo
`/Volumes/workspace/<schema>/dados_vcredit` — copie o caminho exato pelo Catalog, o nome do schema
pode variar). A leitura aqui é feita com **pandas** em vez de `spark.read`: além de contornar a
limitação do Spark serverless com pasta do Workspace, pandas lê **qualquer caminho montado
localmente** (Volume ou Workspace) sem diferença nenhuma de código — por isso este notebook não
precisou mudar quando os CSVs saíram da pasta do Workspace e foram pro Volume.

Regras da camada bronze (ver `CLAUDE.md`):
- **Fiel à origem**: tudo lido como `string`, **sem converter célula vazia em nulo nem interpretar
  nada** (`keep_default_na=False, na_filter=False`) — inclusive o caso de `nota_atendimento` ter linhas
  com a string literal `"NULL"` como texto: aqui ela fica exatamente como está, viraria `null` só na
  silver.
- **Idempotente**: escreve com `overwrite` — rodar de novo dá o mesmo resultado.
- **Rastreável**: toda tabela ganha `_ingestion_ts` e `_source_file`.
- **Sem dado pessoal em output**: a verificação no final só mostra contagens, nunca `display()`/`show()`
  de linhas (⚠️ `clientes` tem CPF/nome/e-mail).

## 1. Configuração

In [ ]:
dbutils.widgets.text("base_path", "")
BASE_PATH = dbutils.widgets.get("base_path")
if not BASE_PATH:
    raise ValueError(
        "Preencha o widget 'base_path' com o caminho do Volume (copie pelo Catalog pra não errar o "
        "nome do schema): '/Volumes/workspace/<schema>/dados_vcredit'. "
        "Também funciona com uma pasta do Workspace, se for o caso: "
        "'/Workspace/Shared/case-final-rocket-lab/dados'."
    )

CATALOGO = "workspace"
SCHEMA_BRONZE = "vcredit_bronze"

# cria o schema de destino se ainda não existir, pra não falhar na primeira execução
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA_BRONZE}")

## 2. Fontes
Arquivo → (tabela de destino, encoding). `clientes.csv` e `Chamados_Hora.CSV` são `ISO-8859-1`; os
demais `UTF-8`.

In [ ]:
SOURCES = {
    "chamados.csv":            ("chamados",            "UTF-8"),
    "Chamados_Hora.CSV":       ("chamados_hora",       "ISO-8859-1"),
    "clientes.csv":            ("clientes",            "ISO-8859-1"),
    "custos.csv":              ("custos",              "UTF-8"),
    "pesquisa_satisfacao.csv": ("pesquisa_satisfacao", "UTF-8"),
    "base_atendentes.csv":     ("base_atendentes",     "UTF-8"),
    "base_motivos.csv":        ("base_motivos",        "UTF-8"),
    "canais.csv":              ("canais",              "UTF-8"),
}

## 3. Leitura (pandas, fiel à origem) e escrita idempotente na bronze

`dtype=str` + `keep_default_na=False` + `na_filter=False`: nenhuma célula vira `NaN`/nulo aqui, tudo
fica exatamente como está no arquivo (inclusive vazio como `""` e a string `"NULL"` como texto) — é o
mais "fiel à origem" possível. Correção/interpretação de conteúdo é papel da silver, não daqui.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

resumo = []

for arquivo, (tabela, encoding) in SOURCES.items():
    caminho = f"{BASE_PATH}/{arquivo}"

    pdf = pd.read_csv(
        caminho,
        dtype=str,
        encoding=encoding,
        keep_default_na=False,
        na_filter=False,
    )
    # padroniza nomes de coluna (remove BOM/espaços à toa, minúsculo) — não muda o conteúdo das células
    pdf.columns = [c.replace("\ufeff", "").strip().lower() for c in pdf.columns]

    df_bronze = (
        spark.createDataFrame(pdf)
        .withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_source_file", F.lit(arquivo))
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{tabela}"
    (
        df_bronze.write
        .mode("overwrite")               # idempotente: roda de novo, mesmo resultado
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )

    linhas = spark.table(destino).count()
    resumo.append((tabela, arquivo, encoding, linhas))
    print(f"{destino}: {linhas:,} linhas")

## 4. Verificação

Contagens esperadas (arquivos originais, ver `docs/achados_analise_geral.md`): chamados 134.474 ·
chamados_hora 131.838 · clientes 47.705 · custos 131.069 · pesquisa_satisfacao 81.021 ·
base_atendentes 25 · base_motivos 14 · canais 4. Essa célula só mostra contagens — nunca linhas
inteiras (⚠️ `clientes` tem CPF/nome/e-mail).

In [ ]:
display(spark.createDataFrame(resumo, ["tabela", "arquivo", "encoding", "linhas"]))

## Checklist antes de comitar / dar push

- [ ] Contagens da seção 4 batem com o esperado (`docs/achados_analise_geral.md`).
- [ ] `base_path` aponta pro Volume (`/Volumes/workspace/<schema>/dados_vcredit`), não mais pra pasta
      `dados` antiga do Workspace — ver `docs/databricks-estrutura.md`.
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail).
- [ ] Rodei duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Se precisou mudar algo, foi por PR revisado — não direto na `main`.